# ATML PA4 — Task 2 (Colab Pro GPU Execution: Parts A, B, C)

Select **one GPU tier** (recommended: **L4**; T4 also works) in `Runtime -> Change runtime type` and keep the same GPU tier across all Task 2 benchmarks. Every part saves resumable outputs straight to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/umairs5/atml-pa4.git'
DATA_REPO_URL = 'https://github.com/AbDu11aHHH/atml-pa4-datasets.git'
DRIVE_TASK2_DIR = '/content/drive/MyDrive/atml-pa4/results/task2'

In [ ]:
%cd /content
!if [ -d atml-pa4/.git ]; then git -C atml-pa4 pull --ff-only; else git clone {REPO_URL} atml-pa4; fi
%cd /content/atml-pa4
!if [ ! -d atml-pa4-datasets/.git ]; then git clone {DATA_REPO_URL} atml-pa4-datasets; fi
!cp configs/local_source_revisions.json atml-pa4-datasets/data/local_source_revisions.json
!nvidia-smi
!python -c "import torch; assert torch.cuda.is_available(), 'No CUDA GPU detected! Select L4 or T4 in Runtime -> Change runtime type.'"
!pip install -q -r requirements.txt

In [ ]:
# Run Task 2 unit tests and dataset asset validation
!python atml-pa4-datasets/scripts/validate_assets.py
!python -m unittest tests/test_task2.py -v

In [ ]:
# Part A: Exact dense, tiled online-softmax, and PyTorch SDPA benchmarks
!python scripts/run_task2_a.py --dataset-root ./atml-pa4-datasets --output-dir {DRIVE_TASK2_DIR}/part_a --repeats 10

In [ ]:
# Part B: MQAR training (B1/B2/B3 across seeds 6304..6306), isolation controls, and T=256..8192 sweep
!python scripts/run_task2_b.py --dataset-root ./atml-pa4-datasets --output-dir {DRIVE_TASK2_DIR}/part_b

In [ ]:
# Part C: Pretrained Qwen2.5-1.5B-Instruct KV compression (C1-C5) across 180 prompts + packed encode/decode bench
!python scripts/run_task2_c.py --dataset-root ./atml-pa4-datasets --output-dir {DRIVE_TASK2_DIR}/part_c

In [ ]:
# Generate summary tables, paired 95% bootstrap CIs (2,000 resamples), and plots
!python scripts/analyze_task2.py --task2-dir {DRIVE_TASK2_DIR} --out-dir {DRIVE_TASK2_DIR}/analysis

import pandas as pd
for name in ('part_a_summary.csv', 'part_b_summary.csv', 'part_c_summary.csv'):
    print('===', name, '===')
    display(pd.read_csv(f'{DRIVE_TASK2_DIR}/analysis/{name}'))

!cd {DRIVE_TASK2_DIR} && zip -rq /content/task2_results.zip part_a/ part_b/env_and_theory.json part_b/inference_sweep.json part_b/runs/ part_c/c1_equivalence_checks.json part_c/c5_packed_encode_decode_bench.json part_c/token_length_audit.json part_c/policies/ part_c/qualitative_masks/ analysis/
!cp /content/task2_results.zip {DRIVE_TASK2_DIR}/task2_results.zip
from google.colab import files
files.download('/content/task2_results.zip')

In [ ]:
# Flush Google Drive writes and disconnect/terminate runtime to save Colab compute units
import time
from google.colab import drive, runtime

time.sleep(15)
drive.flush_and_unmount()
runtime.unassign()